# XDF to CSV: a small worked example

This notebook uses `mock_streams.xdf`, a synthetic recording with Force, heart rate, event markers, and an empty status stream. It contains no participant data. You can replace the path with your own `.xdf` file in Step 2.

**Result:** one CSV per stream, each with an LSL timestamp column and named channels. Separate files preserve each stream's own sample rate.

## 1. Install packages

Use a Python 3.10+ Jupyter kernel. Run this cell once; `%pip` installs into the active kernel. The command-line script needs only `pyxdf`; `pandas` is used here to preview the CSVs.

In [ ]:
%pip install pyxdf==1.17.5 pandas

## 2. Choose the XDF file

Keep `mock_streams.xdf` next to this notebook. For a real recording, replace the value with an absolute path such as `Path(r"C:\\path\\to\\recording.xdf")`.

In [ ]:
from pathlib import Path

xdf_path = Path("mock_streams.xdf")
if not xdf_path.is_file():
    raise FileNotFoundError(f"Change xdf_path to an existing .xdf file: {xdf_path.resolve()}")
print(xdf_path.resolve())

## 3. Read and inspect streams

PyXDF loads numeric samples and string markers. Clock synchronization puts streams on a comparable LSL time basis; disabling dejitter keeps the recorded sample timing. These are LSL seconds, not calendar timestamps.

In [ ]:
import pyxdf

streams, _ = pyxdf.load_xdf(
    str(xdf_path), synchronize_clocks=True, dejitter_timestamps=False
)
for number, stream in enumerate(streams, 1):
    info = stream["info"]
    print(number, info["name"][0], info["type"][0],
          f"{len(stream['time_stamps'])} samples",
          f"{info['channel_count'][0]} channels")

## 4. Export one CSV per stream

The first column is the LSL timestamp. Channel headers include their position, label, and unit where available. A stream with no samples still gets a header-only CSV. The output folder is `<recording>_csv` beside the XDF; the cell stops if that folder already exists so it cannot overwrite an earlier export.

In [ ]:
import csv
import re

output_dir = xdf_path.with_name(f"{xdf_path.stem}_csv")
output_dir.mkdir()
for number, stream in enumerate(streams, 1):
    info = stream["info"]
    name = info["name"][0]
    count = int(info["channel_count"][0])
    channels = info.get("desc", [{}])[0].get("channels", [{}])[0].get("channel", [])
    headers = []
    for index in range(count):
        channel = channels[index] if index < len(channels) else {}
        label = channel.get("label", [f"value_{index + 1}"])[0]
        unit = channel.get("unit", [""])[0]
        headers.append(f"ch{index + 1}_{label}" + (f" [{unit}]" if unit else ""))

    safe_name = re.sub(r"\W+", "_", name).strip("_")[:60] or "stream"
    csv_path = output_dir / f"{number:02d}_{safe_name}.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["lsl_time_s", *headers])
        for timestamp, sample in zip(stream["time_stamps"], stream["time_series"], strict=True):
            if len(sample) != count:
                raise ValueError(f"{name}: expected {count} channels, got {len(sample)}")
            writer.writerow([timestamp, *sample])
    print(f"{csv_path.name}: {len(stream['time_stamps'])} samples")

## 5. Check the CSVs

The example has streams with different row counts. Do not line up rows by index across CSVs; use `lsl_time_s` when relating streams.

In [ ]:
import pandas as pd
from IPython.display import display

for csv_path in sorted(output_dir.glob("*.csv")):
    table = pd.read_csv(csv_path)
    print(f"{csv_path.name}: {len(table)} rows")
    display(table.head())

## 6. Optional: expand the mock JSON markers

The converter preserves string samples exactly. Respyra markers are JSON text, which you can expand into ordinary analysis columns. This example targets the synthetic marker stream; change the filename pattern for your own recording.

In [ ]:
import json

marker_csv = next(output_dir.glob("*Mock_Study_Markers.csv"))
marker_rows = pd.read_csv(marker_csv)
events = pd.json_normalize(marker_rows["ch1_Event"].map(json.loads))
display(pd.concat([marker_rows[["lsl_time_s"]], events], axis=1))

## What to keep

Keep the original XDF as the authoritative recording. The CSVs are convenient flat exports: they retain sample values, timestamps, channel labels, and units, while other XDF metadata is not copied. An empty stream's CSV has only a header and cannot be plotted as a signal. To run the export cell again, rename or remove the previous `<recording>_csv` folder.